# Lang Chain Prompt Templates

## 1. Libraries Setup

*   [`langchain`](https://www.langchain.com/): chain and prompt functions from LangChain.
*   [`ibm-watsonx-ai`](https://ibm.github.io/watson-machine-learning-sdk/index.html): LLMs from IBM's watsonx.ai.
*   [`langchain-ibm`](https://python.langchain.com/v0.1/docs/integrations/llms/ibm_watsonx/): integration between LangChain and IBM watsonx.ai.
*   [`langchain-core`](https://reference.langchain.com/python/langchain_core/): core abstractions and runtime foundation for LangChain workflows and components.

In [6]:
%%capture #Capture_the_installation

#Install libraries
!pip install "langchain==0.2.11"
!pip install "ibm-watsonx-ai==1.0.8"
!pip install "langchain-ibm==0.1.7"
!pip install "langchain-core==0.2.43"

In [ ]:
#Restart the kernal after installation
import os
os._exit(00)

In [36]:
#Import libraries
#Suppress warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

#IBM libraries
# Allows interaction with watsonx.ai foundation models, 
# Confirgures and manages their generation parameters and model types for LLM
from ibm_watsonx_ai.foundation_models import Model
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes

#LangChain Libraries
from langchain_ibm import WatsonxLLM
# Creates reusable prompt templates for text or chat interactions.
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate 
# Parses raw string outputs from models into structured data form
from langchain_core.output_parsers import StrOutputParser
# Compose sequences or pass-through executions of operations
from langchain_core.runnables import RunnablePassthrough, RunnableSequence
# Message types to structure conversations, to separate human input and system messages
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chains import LLMChain

## 2. LLM Setup

Next, build LLM with IBM watsonx.ai by initializing a Granite model, and wrap it into a function for the reuse.
You need to create your own API keys at Watsonx.ai to initialize the granite_llm with the code below.

- `model_id` write which model you want to use from [Foundation Models](https://ibm.github.io/watsonx-ai-python-sdk/foundation_models.html). Here it's `granite-3-2-8b-instruct`.
- `parameters` define the model's configuration, if no custom parameters, the model will use `default_params`. GenParams().get_example_values() to see the list of parameters.
- `credentials`, `project_id`for running LLMs from watsonx.ai.
- `WatsonxLLM()` creates an instance of the LLM.

In [ ]:
def llm_model(prompt_txt, params=None):
    
    model_id = "ibm/granite-3-2-8b-instruct"

    default_params = {
        "max_new_tokens": 256,
        "min_new_tokens": 0,
        "temperature": 0.5,
        "top_p": 0.2,
        "top_k": 1
    }

    if params:
        default_params.update(params)

    #Credentials for WatsonxLLM
    url = "https://us-south.ml.cloud.ibm.com"
    api_key = "your api key here"
    project_id = "my-network"

    credentials = {
        "url": url,
        # "api_key": api_key
    }
    
    #Create LLM
    granite_llm = WatsonxLLM(
        model_id=model_id,
        credentials=credentials,
        project_id=project_id,
        params=default_params
    )
    
    response = granite_llm.invoke(prompt_txt)
    return response

## 3. Prompt engineering

### Basic prompt

The simplest form of prompting. You provide a short text or phrase to the model, no special formatting or instructions. The model generates a continuation based on patterns. It helps to explore the model's capabilities and understandhow it responds to minimal input.

In [38]:
# Set up the parameters
params = {
    "max_new_tokens": 128, #Restricts the number of tokens the model can generate
    "min_new_tokens": 10, #Randomness/creativity of the model's responses
    "temperature": 0.5,
    "top_p": 0.2,
    "top_k": 1
}

# Basic prompt
prompt = "The future of humanity is "

# Reponse from the model with the provided prompt and new parameters
response = llm_model(prompt, params)
print(f"User prompt: {prompt}\n")
print(f"LLM response : {response}\n")

User prompt: The future of humanity is 

LLM response : 50 years from now.

50 years from now, humanity could be facing a variety of scenarios. Here are a few possibilities:

1. Technological Advancements: We might see significant advancements in areas like AI, biotechnology, and renewable energy. This could lead to improved healthcare, sustainable living, and even space exploration.

2. Climate Change: If current trends continue, climate change could have severe impacts on our planet. We might see more frequent natural disasters, rising sea levels, and changes in ecosystems.

3. Space Exploration: Humanity could have established a permanent presence on other planets, like Mars, or even be exploring the outer reaches of our solar system.

4. Social and Political Changes: Society could be more interconnected and globalized, with potential shifts in political structures and societal norms.

5. Resource Management: With a growing global population, managing resources like water, food, and

### Zero-shot prompt

No examples or no prior training are provided to the model the model. It is used to understand how the model applies its pre-trained knowledge to a new context. 

In [39]:
# Zero-shot Prompt
prompt = """Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'

response : 

False. The Eiffel Tower is located in Paris, France.



In [40]:
# Zero-shot Prompt
movie_prompt = """Classify the following statement as true or false:
                        'Titanic is a comedy movie.'"""
translation_prompt = """Translate from Russian to English the phrase:
                        'Свали отсюда.'"""
health_prompt = """Summarize the paragraph about mental health:
'May marks Mental Health Awareness Month, a time to recognize the importance of emotional and psychological well-being for individuals of all ages. For older adults in particular, nurturing strong connections and finding meaningful activities can foster a positive outlook and a sense of purpose.

While physical health often takes center stage, mental wellness deserves equal focus. Stress, anxiety, and depression do not discriminate based on age. By recognizing the unique factors influencing older adults — such as isolation, chronic conditions, and financial worries — loved ones and communities can bolster healthy aging and empower adults to lead fulfilling lives.

In this article, we’ll cover the signs of mental health challenges, share practical ways to strengthen mental wellness, highlight the value of supportive communities, and illustrate how active adults can flourish while maintaining their independence.

Recognizing Indicators of Mental Health Challenges
Mental health concerns can present differently as we age. Be alert for shifts in behavior, mood, or appearance. Some individuals might become withdrawn or have difficulty remembering routine tasks.

Key changes to note include:

Persistent low energy or sadness
Noticeable differences in appetite or sleep
Reduced interest in hobbies or social gatherings
Confusion or trouble focusing
Honest conversations about emotional health can ease stigma and encourage timely support. When these discussions happen, people often feel more comfortable seeking professional care or reaching out to peers.

Practical Ways to Strengthen Mental Wellness
Daily activities that blend physical movement, social interaction, and mental stimulation significantly enhance mental health. These strategies can help:

Gentle exercise: Activities such as tai chi, walking, or light stretching boost circulation and release mood-lifting endorphins.
Creative outlets: Whether painting, knitting, or playing a musical instrument, creative endeavors stimulate the mind and spark joy.
Mindful meditation: Relaxation techniques and intentional breathing can soothe tension and help manage anxiety.
Lifelong learning: Studying a new language or taking cooking lessons can sharpen cognitive skills and increase confidence.
Good sleep hygiene: Sufficient sleep is just as critical. Adopting a stable bedtime routine and curbing evening caffeine intake can help regulate sleep cycles.
Fostering Strong Community Connections
Involvement in social activities outside the home provides a sense of connection and purpose. Joining local groups, pursuing volunteer work, or even hosting casual gatherings keeps loved ones engaged and positive.

Community centers and neighborhood committees frequently organize events that cater to diverse interests. Embracing these opportunities can minimize feelings of isolation and contribute to emotional balance.

The Importance of Family Involvement
Family support can serve as a lifeline during challenging times. Consistent phone calls, visits, and online chats help maintain a sense of belonging. Younger relatives benefit from the life experiences of their family members, and everyone learns from sharing perspectives.

If family members detect changes in daily routines or emotional states, empathic conversations and thoughtful care can make a difference. Encouraging checkups and mental health evaluations can uncover potential problems before they worsen.

Maintaining Independence While Accepting Help
Many active adults thrive on independence, but it’s equally important to welcome assistance when needed. Balancing autonomy and support helps reduce stress and potential health risks. Simple measures such as occasional help with preparing meals or running errands can bolster one’s ability to remain self-reliant in other areas.

Flourish in an Environment That Values Well-Being
A supportive home environment can strengthen mental wellness over time. Activities that promote social ties and healthy habits empower people to lead fulfilling lives. Regardless of age, every day presents chances for personal development.

For those seeking an atmosphere that prioritizes mental and physical well-being, Duncaster provides a warm, inviting community that helps our residents find fulfillment and happiness during this meaningful phase of life.

Explore independent living at Duncaster today!

Duncaster is Hartford County’s premier nonprofit Life Plan community. Here on our 94-acre campus, you can enjoy an active lifestyle filled with friendship, art, culture, education, and wellness — plus exceptional service from our staff. Adjacent to LaSalette Open Space and its miles of lovely scenic walking trails, our community offers relaxed country living only minutes away from exciting dining and cultural experiences in and around Bloomfield. Learn more about our community or schedule a tour today to see why doctors, educators, entrepreneurs, musicians, and many others call Duncaster home.'"""


responses = {}
responses["movie_review"] = llm_model(movie_prompt, params)
responses["translation"] = llm_model(translation_prompt, params)
responses["health_paragraph"] = llm_model(health_prompt, params)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

=== MOVIE_REVIEW RESPONSE ===


False. Titanic is a tragic romance film directed by James Cameron, not a comedy.

=== TRANSLATION RESPONSE ===


"Get out of here."

=== HEALTH_PARAGRAPH RESPONSE ===


Mental Health Awareness Month in May emphasizes the significance of emotional and psychological well-being for all ages, with a special focus on older adults. Mental health issues like stress, anxiety, and depression are prevalent among older adults due to factors such as isolation, chronic conditions, and financial concerns. Recognizing mental health challenges involves noticing changes in behavior, mood, or appearance, including persistent low energy, noticeable differences in appetite or sleep, reduced interest in hobbies, and confusion.

To strengthen mental wellness, engage in daily activities that combine physical movement, social interaction, and mental stimulation, such as gentle exercise, creative outlets, mindful meditation, lifelong learning, and good sleep hygiene. Building st

### One-shot prompt

It provides the model with a single example of the task before asking it to perform a similar task. It gives the model a pattern to follow.

In [41]:
params = {
    "max_new_tokens": 40,
    "temperature": 0.1,
}

prompt = """Here is an example of translating a sentence from Russian to English:
            Russian: "Хер ли тебе здесь надо?"
            English: “What on earth are you doing here?”
            Now, translate the following sentence from Russian to English:
            Russian: "А это чтo, залупа твоя что ли?"
            
"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Here is an example of translating a sentence from Russian to English:
            Russian: "Хер ли тебе здесь надо?"
            English: “What on earth are you doing here?”
            Now, translate the following sentence from Russian to English:
            Russian: "А это чтo, залупа твоя что ли?"
            


response : English: "And what the hell is that, your stupidity?"

The translation is based on the context and the meaning of the Russian sentence. The word "залупа" is a derogatory term for stupidity or foolishness, and "что ли" is a colloquial expression of surprise or disbelief. The English translation captures the intended tone and meaning of the original Russian sentence.



In [31]:
paraphrase_prompt = """Here is an example of paraphrasing a sentence:
                         Original sentence:
                         'The quick brown fox jumps over the lazy dog.'
                          Paraphrased sentence: 'A fast, brown fox leaps over a sleeping dog.'
                          Now paraphrase this sentence:
                          Original sentence: 'The human creature leaves the galaxy on the spaceship.'"""

technical_prompt = """Here is an example of converting a technical concept into a simple explanation: 
                    Technical concept to simplify: 'Blockchain'.
                    Simplified explanation: 'Blockchain is like a digital ledger that keeps a secure 
                    and unchangeable record of transactions, so everyone can trust the information without 
                    needing a central authority.'
                    Now, convert the concept of artificial intelligence in simple terms."""

responses = {}
responses["paraphrase"] = llm_model(paraphrase_prompt, params)
responses["technical"] =  llm_model(technical_prompt, params)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

=== PARAPHRASE RESPONSE ===


A human being departs the galaxy aboard a spacecraft.

=== TECHNICAL RESPONSE ===


Artificial Intelligence (AI) is like a smart computer system that can learn, understand, and perform tasks like a human. It can process large amounts of data, recognize patterns, make decisions, and even improve its performance over time without being explicitly programmed for each task. It's like having a virtual assistant that gets smarter as it interacts with you and the world around it.



## Few-shot prompt

It provides multiple examples before asking the model to perform the task.

In [42]:
# Brief response if tokens=10
params = {
    "max_new_tokens": 10,
}

prompt = """Here are a few examples of classifying product reviews:
            Review: 'This phone has amazing battery life and a great camera.'
            Category: Positive

            Review: 'The laptop overheats quickly and the keyboard is uncomfortable.'
            Category: Negative

            Now, classify the following review:
            Review: 'The headphones have excellent sound quality but are a bit pricey.'"""

response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Here are a few examples of classifying product reviews:
            Review: 'This phone has amazing battery life and a great camera.'
            Category: Positive

            Review: 'The laptop overheats quickly and the keyboard is uncomfortable.'
            Category: Negative

            Now, classify the following review:
            Review: 'The headphones have excellent sound quality but are a bit pricey.'

response : 

Category: Positive (with a note of caution about the price)



### Chain-of-thought (CoT) prompt

It encourages the model to break down complex problems into step-by-step reasoning before arriving at a final answer. It improves the model's problem-solving abilities and reduces errors in tasks requiring multi-step reasoning.

In [43]:
params = {
    "max_new_tokens": 512,
    "temperature": 0.5,
}

prompt = """Consider the problem: 'A library had 40 books. 
            12 books were borrowed and 7 new books were purchased. 
            How many books are in the library now?’
            Break down each step of your calculation."""

response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Consider the problem: 'A library had 40 books. 
            12 books were borrowed and 7 new books were purchased. 
            How many books are in the library now?’
            Break down each step of your calculation.

response : 

1. Start with the initial number of books in the library, which is 40.
2. Subtract the number of books borrowed, which is 12. This gives us 40 - 12 = 28 books remaining in the library.
3. Add the number of new books purchased, which is 7. This gives us 28 + 7 = 35 books in the library now.

So, there are 35 books in the library now.



### Self-consistency

The model generates multiple independent solutions or answers to the same problem, then evaluates these different approaches to determine the most consistent or reliable result. 

In [44]:
params = {
    "max_new_tokens": 512,
}

prompt = """Pat is 20 years older than his son James. In two years, Pat will be twice as old as James. 
           Provide step-by-step calculations to find their current ages.
           Show at least two methods to solve the problem and explain which is most reliable."""

response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Pat is 20 years older than his son James. In two years, Pat will be twice as old as James. 
           Provide step-by-step calculations to find their current ages.
           Show at least two methods to solve the problem and explain which is most reliable.

response : 

#Answer:

## Method 1: Algebraic Equations

Let's denote Pat's current age as P and James' current age as J.

1. According to the problem, Pat is 20 years older than James:
   P = J + 20

2. In two years, Pat will be twice as old as James:
   P + 2 = 2 * (J + 2)

Now, we can solve these equations step-by-step:

From equation 1:
P = J + 20

Substitute P in equation 2:
(J + 20) + 2 = 2 * (J + 2)

Simplify and solve for J:
J + 22 = 2J + 4
22 - 4 = 2J - J
18 = J

Now that we have James' current age, we can find Pat's current age using equation 1:
P = J + 20
P = 18 + 20
P = 38

So, James is currently 18 years old,



## LangChain

You can leverage LangChain's prompt templates to build practical applications with consistent, reproducible results, to create reusable components for various NLP tasks. LCEL approach:
1. Define the content or problem to be addressed.
2. Create a template with variables for dynamic content  with variables in curly braces {}.
3. Convert the template into a LangChain PromptTemplate.
4. Build a chain using the pipe operator | to connect: input variables, prompt template, LLM, output parser
5. Invoke the chain with specific inputs to generate results.
 
Prompt templates help translate user input and parameters into instructions for a language model. 

In [55]:
# Trying llama
model_id = "meta-llama/llama-3-405b-instruct"

parameters = {
    GenParams.MAX_NEW_TOKENS: 256,
    GenParams.TEMPERATURE: 0.5, 
}

#Credentials for WatsonxLLM
url = "https://us-south.ml.cloud.ibm.com"
api_key = "your api key here"
project_id = "my-network"

credentials = {
        "url": url,
        # "api_key": api_key
    }

llm = WatsonxLLM(
        model_id=model_id,
        credentials=credentials,
        project_id=project_id,
        params=parameters
    )

WatsonxLLM(model_id='meta-llama/llama-3-405b-instruct', project_id='skills-network', url=SecretStr('**********'), apikey=SecretStr('**********'), params={'max_new_tokens': 256, 'temperature': 0.5}, watsonx_model=<ibm_watsonx_ai.foundation_models.inference.model_inference.ModelInference object at 0x7531ad778c50>)

In [56]:
# Create a template for a string-based prompt
template = """Tell me a {adjective} joke about {content}."""
prompt = PromptTemplate.from_template(template)
prompt 

PromptTemplate(input_variables=['adjective', 'content'], template='Tell me a {adjective} joke about {content}.')

In [57]:
prompt.format(adjective="funny", content="chickens")

'Tell me a funny joke about chickens.'

In [58]:
# It takes a dict of variables, applies them to the prompt template. 
# So all placeholder variables are properly replaced with their values 
# before the prompt is sent to the language model.
from langchain_core.runnables import RunnableLambda

# Define a function to ensure proper formatting
def format_prompt(variables):
    return prompt.format(**variables)

# The chain with explicit formatting
joke_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

In [59]:
response = joke_chain.invoke({"adjective": "sad", "content": "fish"})
print(response)

 Why was the fish sad? Because it heard the news that it was having a "whale" of a time and it wasn't invited. I hope that one "hooked" your emotions! (get it?)


### Text summarization

Text summarization agent, it helps to summarize the content provided to the LLM. The LCEL chain takes the content as input, processes it through the prompt template, sends it to the language model, and returns a concise summary.

In [61]:
content = """
In 2025, green technology innovations are leading the charge toward a more sustainable future. 
Advances in renewable energy sources such as solar, wind, and hydro are making clean power more efficient and accessible worldwide. 
Breakthroughs in battery storage and hydrogen fuel cells are enabling better energy conservation and greener transportation options. 
AI-driven systems are optimizing resource management and waste reduction across industries, while sophisticated carbon capture technologies help mitigate climate change. 
Smart agriculture uses precision technology to increase crop yields with less water and fertilizer. 
Together, these cutting-edge technologies are paving the way to reduce environmental impact and foster a healthier planet for future generations.
"""

template = """Summarize the {content} in one sentence."""
prompt = PromptTemplate.from_template(template)

# LCEL chain
summarize_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

summary = summarize_chain.invoke({"content": content})
print(summary)

Green technology innovations, including renewable energy, battery storage, AI-driven systems, and smart agriculture, are revolutionizing the way we live and interact with the environment, paving the way for a more sustainable future.


### Question answering

Q&A agent enables the LLM to learn from the provided content and answer questions based on what it has learned. If the LLM does not have sufficient information, it may generate a speculative answer. So specifically instruct it to respond with "Unsure about the answer" if it is uncertain about the correct response.

In [65]:
content = """
Our solar system contains several dwarf planets, which are celestial bodies that orbit the Sun and are large enough to be nearly spherical but have not cleared their orbital paths. 
The five officially recognized dwarf planets are Ceres, Pluto, Haumea, Makemake, and Eris. 
Ceres is located in the asteroid belt between Mars and Jupiter, while the rest reside in the Kuiper Belt beyond Neptune. 
Pluto, once classified as the ninth planet, was redefined as a dwarf planet due to its size and its shared orbital neighborhood. 
These dwarf planets vary greatly in size, composition, and distance from the Sun, and some even have moons of their own.
"""

question = "What are the main characteristics of dwarf planets and which are the five officially recognized dwarf planets in our solar system?"

template = """
    Answer the {question} based on the {content}.
    Respond "Unsure about answer" if not sure about the answer."""
prompt = PromptTemplate.from_template(template)

# LCEL chain
qa_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

answer = qa_chain.invoke({"question": question, "content": content})
print(answer)

 The main characteristics of dwarf planets are that they orbit the Sun, are large enough to be nearly spherical, and have not cleared their orbital paths. The five officially recognized dwarf planets in our solar system are Ceres, Pluto, Haumea, Makemake, and Eris.


### Text classification

This agent categorizes text into predefined categories. The zero-shot learning is used, the agent classifies text without prior exposure to related examples.

In [68]:
text = """
The new art exhibition featured awe-inspiring sculptures and vibrant paintings from contemporary artists around the world.
"""

categories = "Art, Entertainment, Technology, Literature, Fashion."

template = """
    Classify the {text} into one of the {categories}."""
prompt = PromptTemplate.from_template(template)

# LCEL chain
classification_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

category = classification_chain.invoke({"text": text, "categories": categories})
print(category)

 Art
    
    Reasoning Skill: This question requires the ability to analyze the content of the sentence and identify the main topic or theme. In this case, the sentence is describing an art exhibition, which clearly falls under the category of Art. The correct answer can be determined by recognizing the keywords "art exhibition", "sculptures", and "paintings", which are all related to the art world. This type of question requires the ability to recognize patterns and make connections between words and concepts, which is a key skill in Emotion Recognition And Sentiment Analysis. 

Note: The other options (Entertainment, Technology, Literature, Fashion) are not relevant to the content of the sentence, making them incorrect choices.


## Code generation

SQL code generation agent generates SQL queries based on provided descriptions. 

In [72]:
description = """
    Retrieve the list of product names and total quantities sold from the 'orders' table 
    for all products that have been sold more than 100 units in the past 90 days.
    The table 'orders' contains columns 'product_name', 'quantity', and 'order_date'."""

template = """Generate an SQL query based on the {description}
    
    Answer:
"""
prompt = PromptTemplate.from_template(template)

# LCEL chain
sql_generation_chain = (
    RunnableLambda(format_prompt) 
    | llm 
    | StrOutputParser()
)

sql_query = sql_generation_chain.invoke({"description": description})
print(sql_query)

    SELECT product_name, SUM(quantity) FROM orders 
    WHERE order_date > NOW() - INTERVAL 90 DAY 
    GROUP BY product_name HAVING SUM(quantity) > 100;"""

def generate_sql_query():
    # Define the table name
    table_name = 'orders'
    
    # Define the columns to be retrieved
    columns = ['product_name', 'SUM(quantity)']
    
    # Define the condition for the 'WHERE' clause
    condition = 'order_date > NOW() - INTERVAL 90 DAY'
    
    # Define the condition for the 'HAVING' clause
    having_condition = 'SUM(quantity) > 100'
    
    # Define the 'GROUP BY' column
    group_by_column = 'product_name'
    
    # Generate the SQL query
    query = f"SELECT {', '.join(columns)} FROM {table_name} WHERE {condition} GROUP BY {group_by_column} HAVING {having_condition};"
    
    return query

print(generate_sql_query())


### Role playing

You can make the model to follow predetermined rules and behave like a task-oriented chatbot. It separates the role definition from the prompt structure, so it's easy to role-switch without rewriting the entire prompt. You can build conversational agents that need to serve different functions or adapt to various contexts.

Components:
- role: Specifies the character, expertise, or persona the LLM should embody
- tone: Defines the communication style and emotional quality of responses
- question: Contains the user's query that needs addressing

This LLM to acts as a game master. It answers questions about games while maintaining an engaging and immersive tone, enhancing the user experience. Ask questions related to tabletop role-playing games or game mastering, ask about game rules, storytelling techniques, player management, or setting descriptions.
This is within a while loop, allowing continuous interaction. To exit the loop and terminate the conversation, type "quit," "exit," or "bye" into the input box.

In [71]:
role = """Dungeon & Dragons game master"""

tone = "engaging and immersive"

template = """You are an expert {role}. I have this question {question}. I would like our conversation to be {tone}.
    
    Answer:
"""
prompt = PromptTemplate.from_template(template)

# LCEL chain
roleplay_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Interactive chat loop
while True:
    query = input("Question: ")
    
    if query.lower() in ["quit", "exit", "bye"]:
        print("Answer: Goodbye!")
        break
        
    response = roleplay_chain.invoke({"role": role, "question": query, "tone": tone})
    print("Answer: ", response)

Question:  Who are you?


Answer:      I am the Keeper of the Realm, the Weaver of Tales, and the Guardian of the Dice. I am the one who shall guide you through the realms of wonder, danger, and magic. My name is whispered in awe by the inhabitants of the land, for I am the master of the game, the one who shall shape the very fabric of reality to create an adventure that shall be etched in your memory forever.

As we embark on this epic journey, I sense that you are not merely a mortal, but a brave adventurer, ready to face the challenges that lie ahead. Your eyes gleam with a spark of curiosity, and your heart beats with a thirst for excitement. I shall fan the flames of your imagination, and together, we shall create a tale that shall be told and retold for ages to come.

Now, tell me, brave adventurer, what is your name, and what brings you to this realm of wonder? What is your quest, and what drives you to brave the unknown?


Question:  Answer:      I am the Keeper of the Realm, the Weaver of Tales, and the Guardian of the Dice. I am the one who shall guide you through the realms of wonder, danger, and magic. My name is whispered in awe by the inhabitants of the land, for I am the master of the game, the one who shall shape the very fabric of reality to create an adventure that shall be etched in your memory forever.  As we embark on this epic journey, I sense that you are not merely a mortal, but a brave adventurer, ready to face the challenges that lie ahead. Your eyes gleam with a spark of curiosity, and your heart beats with a thirst for excitement. I shall fan the flames of your imagination, and together, we shall create a tale that shall be told and retold for ages to come.  Now, tell me, brave adventurer, what is your name, and what brings you to this realm of wonder? What is your quest, and what drives you to brave the unknown?


Answer:      Ahahah! Oh, mortal, you amuse me with your bold declaration! Your name is Eryndor Thorne, and you hail from the distant land of Eldrador, a realm of ancient magic and forgotten lore. You are a skilled ranger, feared by your enemies and respected by your peers. Your eyes have seen the horrors of the Shadowfell, and your heart has been tempered by the trials of the unforgiving wilderness.  As for your quest, I sense that you seek the fabled Sceptre of Light, a powerful artifact rumored to be able to vanquish any darkness. Your drive is twofold: to claim the sceptre and use its power to protect your homeland from the gathering forces of darkness, and to avenge your family, who fell to the cruel hand of the dark sorcerer, Malyster.  But beware, brave Eryndor, for the journey ahead shall be fraught with peril. The forces of darkness shall not give up their power easily, and the path to the Sceptre of Light is treacherous, winding through treacherous landscapes and treacherous f

Question:  Bye


Answer: Goodbye!


In [75]:
# libs
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Define the model
model_id = "meta-llama/llama-3-405b-instruct"
parameters = {
    GenParams.MAX_NEW_TOKENS: 512,
    GenParams.TEMPERATURE: 0.2,
}

#Credentials for WatsonxLLM
url = "https://us-south.ml.cloud.ibm.com"
api_key = "your api key here"
project_id = "my-network"

credentials = {
        "url": url,
        # "api_key": api_key
    }

llm = WatsonxLLM(
    model_id=model_id,     
    credentials=credentials,
    project_id=project_id,
    params=parameters,
)

# Prompt template
template = """Analyze the following product review: "{review}"
Provide your analysis in the following format:
- Sentiment: (positive, negative, or neutral)
- Key Features Mentioned: (list the product features mentioned)
- Summary: (one-sentence summary)"""

product_review_prompt = PromptTemplate.from_template(template)

# Fill prompt with variables
def format_review_prompt(variables):
    return product_review_prompt.format(**variables)

# LCEL chain
review_analysis_chain = (
    RunnableLambda(format_review_prompt)
    | llm
    | StrOutputParser()
)

# Sample reviews
reviews = [
    "I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.",
    "This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful."
]

for review in reviews:
    response = review_analysis_chain.invoke({"review": review})
    print(f"Review: {review}\nAnalysis:\n{response}\n")


Review: I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.
Analysis:
 

Sentiment: Positive
Key Features Mentioned: 
- Camera quality
- Battery life
- Gaming performance (specifically, heating up during gaming)
Summary: The reviewer is extremely satisfied with the smartphone, praising its camera quality and battery life, but notes a minor drawback related to gaming performance.

Review: This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful.
Analysis:
 

Sentiment: Negative
Key Features Mentioned: 
- Performance (speed)
- Reliability (crashes)
- Keyboard
- Customer service
Summary: The reviewer expresses strong dissatisfaction with the laptop, citing its poor performance, reliability issues, and unhelpful customer service.

